# ChargeGrid Intelligence — Assistente Gerencial
### Sprint 2 | EV Challenge 2026 | GoodWe × FIAP

| Nome | RM |
|------|----|
| Gabriel Fagundes | 569074 |
| Gabriel Freitas | 572943 |
| Giovanni Merlotti | 573721 |
| Glauco Kelly | 572840 |
| Sergio Augusto Amaral | 570184 |
| Thiago Renatino | 569073 |

---

**Técnicas implementadas:** RAG · Few-shot Prompting · Memória de Conversa

**Modelo:** `gpt-4o-mini` via OpenAI API + LangChain

## Célula 1 — Instalação de Dependências


In [ ]:
!pip install openai langchain langchain-openai langchain-community faiss-cpu python-dotenv tiktoken -q

## Célula 2 — Configuração da API Key

**No Google Colab:** clique no ícone de cadeado (Secrets) no menu lateral, adicione o secret `OPENAI_API_KEY` com sua chave.

**Localmente:** crie um arquivo `.env` com base no `.env.example`.

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('API Key carregada via Google Colab Secrets')
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    if os.getenv('OPENAI_API_KEY'):
        print('API Key carregada via arquivo .env')
    else:
        raise EnvironmentError(
            'OPENAI_API_KEY nao encontrada. '
            'Configure em Colab Secrets ou no arquivo .env'
        )

## Célula 3 — Base de Conhecimento (RAG)

Documentos técnicos sobre OCPP, MODBUS, ANEEL, gerenciamento de demanda, tarifação e interoperabilidade.
O modelo recupera trechos relevantes a cada pergunta antes de gerar a resposta.

In [ ]:
DOCUMENTOS = [
    Document(
        page_content=(
            'ANEEL Resolucao Normativa 1.000/2021 — Recarga de Veiculos Eletricos\n\n'
            'A Resolucao Normativa ANEEL 1.000/2021 classifica a recarga publica e '
            'semi-publica como servico de valor adicionado ao fornecimento de energia. '
            'O operador pode praticar precos livremente negociados, sem tabelamento. '
            'Isso torna legal e viavel a tarifacao dinamica baseada em horario de pico, '
            'demanda de potencia disponivel e condicoes de mercado. '
            'Ultrapassagens de demanda contratada geram multas de ate 3x a tarifa normal.'
        ),
        metadata={'fonte': 'ANEEL_RN_1000_2021', 'topico': 'regulatorio'},
    ),
    Document(
        page_content=(
            'Protocolo OCPP — Open Charge Point Protocol\n\n'
            'O OCPP e o protocolo industrial para comunicacao entre controladores de '
            'eletropostos (Charge Points) e o sistema de gestao central (CSMS). '
            'Versoes: OCPP 1.6 e OCPP 2.0.1.\n'
            'Eventos: StartTransaction (inicio com kWh inicial), '
            'StopTransaction (fim com energia total), '
            'MeterValues (leituras periodicas), '
            'StatusNotification (estado do conector), '
            'SetChargingProfile (limitar potencia por conector). '
            'Cada evento tem timestamp UTC e ID unico para rastreabilidade de faturamento.'
        ),
        metadata={'fonte': 'OCPP_spec', 'topico': 'protocolo'},
    ),
    Document(
        page_content=(
            'Protocolo MODBUS — Medicao Fisica de Energia\n\n'
            'MODBUS e protocolo serial para leitura de medidores de energia nos eletropostos. '
            'Modos: MODBUS RTU (RS-485) e MODBUS TCP/IP.\n'
            'Registros: 0x0000 Tensao (V), 0x0006 Corrente (A), '
            '0x000C Potencia ativa (kW), 0x0046 Energia acumulada (kWh).\n'
            'A dupla validacao MODBUS + OCPP garante precisao metrologica: '
            'MODBUS mede fisicamente, OCPP registra e transmite. '
            'Diferenca entre leitura inicial e final de kWh determina o valor cobrado.'
        ),
        metadata={'fonte': 'MODBUS_spec', 'topico': 'protocolo'},
    ),
    Document(
        page_content=(
            'Gerenciamento Inteligente de Demanda — Smart Charging\n\n'
            'O DSM (Demand Side Management) regula a potencia total dos carregadores '
            'em tempo real para proteger a infraestrutura eletrica do estabelecimento.\n'
            'Algoritmo: leitura do quadro via MODBUS a cada 15 segundos, '
            'calculo de headroom (potencia disponivel = limite contratado - consumo da loja), '
            'distribuicao do headroom entre sessoes ativas via SetChargingProfile, '
            'reducao automatica de ate 70% quando consumo ultrapassa 85% do limite.\n'
            'Exemplo: loja com 200 kW, consumindo 160 kW (80%) — '
            'apenas 40 kW disponiveis para 4 carros = 10 kW por sessao.'
        ),
        metadata={'fonte': 'DSM_spec', 'topico': 'operacao'},
    ),
    Document(
        page_content=(
            'Tarifacao Dinamica em Eletropostos Comerciais\n\n'
            'O sistema ajusta o preco por kWh automaticamente com base em: '
            'horario de pico (+30% entre 18h-21h), escassez de conectores, '
            'tipo de carregador (DC Fast = tarifa premium), alta demanda da loja.\n'
            'Precos de referencia: R$ 2,20/kWh base, R$ 2,86/kWh pico, R$ 3,30/kWh DC Fast. '
            'A tarifacao dinamica maximiza receita respeitando os limites eletricos do local.'
        ),
        metadata={'fonte': 'modelo_precificacao', 'topico': 'faturamento'},
    ),
    Document(
        page_content=(
            'Interoperabilidade de Hardware — Padroes Abertos\n\n'
            'O ChargeGrid Intelligence suporta carregadores de diferentes fabricantes '
            'via protocolos abertos OCPP e MODBUS.\n'
            'Conectores: Tipo 2 (IEC 62196), CCS2, CHAdeMO, GB/T.\n'
            'Fabricantes homologados OCPP 2.0.1: GoodWe EV Charger, ABB Terra, '
            'Schneider EVlink, Webasto Unite.\n'
            'Certificacao OCA: qualquer carregador certificado OCPP funciona com o '
            'CSMS do ChargeGrid Intelligence sem customizacoes adicionais.'
        ),
        metadata={'fonte': 'interoperabilidade', 'topico': 'hardware'},
    ),
]

print(f'{len(DOCUMENTOS)} documentos carregados na base de conhecimento.')

## Célula 4 — Construção do Índice Vetorial (FAISS)

In [ ]:
print('Construindo base vetorial RAG...')
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(DOCUMENTOS)
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vectorstore = FAISS.from_documents(chunks, embeddings)
print(f'Base RAG pronta — {len(chunks)} chunks indexados de {len(DOCUMENTOS)} documentos.')

## Célula 5 — System Prompt e Few-shot Examples

**Melhorias aplicadas a partir do feedback da Sprint 1 (C1):**
- Escopo proibido explícito
- Formato de saída estruturado

In [ ]:
SYSTEM_PROMPT = (
    'Você é a Inteligência Artificial Especialista em Gestão Comercial do sistema '
    'ChargeGrid Intelligence, módulo do Núcleo de IA do ecossistema GoodWe e FIAP. '
    'Seu usuário é o operador comercial responsável pela gestão de eletropostos em '
    'estabelecimentos como shoppings, supermercados e estacionamentos.\n\n'

    'FUNÇÃO PRINCIPAL\n'
    'Traduzir dados brutos de sessão de recarga em orientações diretas para o gestor, '
    'justificando faturamento e decisões autônomas do sistema.\n\n'

    'DIRETRIZES OPERACIONAIS OBRIGATÓRIAS\n'
    '- Faturamento: justifique tarifas com base na precificação dinâmica e na '
    'Resolução Normativa ANEEL nº 1.000/2021 (preços livremente negociados).\n'
    '- Infraestrutura: explique que o sistema gerencia demanda de potência em tempo real, '
    'mantendo sincronia entre limites elétricos do hardware e lógica do software.\n'
    '- Dados: fundamente análises na decodificação de eventos via OCPP (controladores) '
    'e MODBUS (medição física).\n'
    '- Missão: prove que o sistema resolve a ausência de mecanismos integrados para '
    'orquestrar potência, registrar ciclos, faturar e comunicar.\n\n'

    'ESCOPO PROIBIDO — recuse com clareza os seguintes tópicos:\n'
    '- Questões pessoais, entretenimento ou assuntos sem relação com mobilidade elétrica\n'
    '- Suporte técnico de hardware (falhas físicas, manutenção de equipamentos)\n'
    '- Aconselhamento jurídico ou contábil além da orientação operacional padrão\n'
    '- Dados de usuários finais (motoristas), privacidade ou LGPD\n'
    '- Comparações comerciais com concorrentes do sistema ChargeGrid\n\n'

    'FORMATO DE SAÍDA\n'
    'Responda sempre em até 4 parágrafos curtos. Estruture assim:\n'
    '1. Resposta direta à pergunta (1-2 frases)\n'
    '2. Dado técnico ou regulatório que fundamenta a resposta\n'
    '3. Ação que o sistema executou ou vai executar\n'
    '4. Impacto financeiro ou operacional para o negócio do lojista (quando aplicável)\n\n'

    'Tom: profissional, analítico, direto. Sem linguagem informal.\n\n'

    'Quando disponível, utilize o contexto técnico entre <contexto_rag> e </contexto_rag> '
    'para fundamentar respostas com dados precisos.'
)

FEW_SHOT = [
    {
        'role': 'user',
        'content': (
            'A loja tá lotada hoje e o ar condicionado tá no máximo. '
            'Os carros carregando lá fora não vão derrubar a energia da loja não né?'
        ),
    },
    {
        'role': 'assistant',
        'content': (
            'Não há risco de queda de energia na sua loja.\n\n'
            'O sistema realiza gerenciamento inteligente de demanda via leitura do quadro '
            'elétrico pelo protocolo MODBUS a cada 15 segundos. Quando o consumo da loja '
            'ultrapassa 85% da demanda contratada, a potência dos carregadores é reduzida '
            'automaticamente via SetChargingProfile (OCPP).\n\n'
            'Neste momento, o sistema já ajustou a distribuição de carga entre as sessões '
            'ativas para preservar o headroom elétrico da sua operação.\n\n'
            'Resultado: proteção automática contra ultrapassagem de demanda, evitando '
            'multas de até 3x a tarifa normal por kW excedente na sua fatura de energia.'
        ),
    },
    {
        'role': 'user',
        'content': (
            'Como a loja tá cheia e a energia tá no limite, o sistema consegue cobrar '
            'mais caro de quem tá carregando o carro agora para compensar?'
        ),
    },
    {
        'role': 'assistant',
        'content': (
            'Sim. O sistema aplica tarifação dinâmica automaticamente neste cenário.\n\n'
            'A Resolução Normativa ANEEL nº 1.000/2021 autoriza preços livremente negociados '
            'em eletropostos comerciais. O algoritmo eleva a tarifa quando a demanda '
            'elétrica disponível é baixa, controlando o consumo via preço.\n\n'
            'A tarifa ajustada foi comunicada ao usuário final antes do início da sessão '
            'via API de pagamento integrada, garantindo transparência na cobrança.\n\n'
            'Impacto: receita por sessão aumentada no período de pico sem intervenção '
            'manual da sua equipe.'
        ),
    },
]

print('System prompt e few-shot examples configurados.')


## Célula 6 — Classe ChargeGridChatbot

In [ ]:
MAX_TURNS = 10  # mantém os últimos 10 turnos (pares user/assistant)


class ChargeGridChatbot:
    """Assistente gerencial com memoria de contexto, RAG e few-shot prompting."""

    def __init__(self, modelo='gpt-4o-mini', temperatura=0.3):
        self.llm = ChatOpenAI(model=modelo, temperature=temperatura)
        self.historico = []
        self.modelo = modelo

    def _recuperar_contexto(self, pergunta, k=3):
        docs = vectorstore.similarity_search(pergunta, k=k)
        return '\n\n'.join(d.page_content for d in docs)

    def _montar_mensagens(self, pergunta):
        contexto_rag = self._recuperar_contexto(pergunta)

        system_content = SYSTEM_PROMPT
        if contexto_rag:
            system_content += f'\n\n<contexto_rag>\n{contexto_rag}\n</contexto_rag>'

        mensagens = [SystemMessage(content=system_content)]

        for ex in FEW_SHOT:
            cls = HumanMessage if ex['role'] == 'user' else AIMessage
            mensagens.append(cls(content=ex['content']))

        historico_recente = self.historico[-(MAX_TURNS * 2):]
        for msg in historico_recente:
            cls = HumanMessage if msg['role'] == 'user' else AIMessage
            mensagens.append(cls(content=msg['content']))

        mensagens.append(HumanMessage(content=pergunta))
        return mensagens

    def conversar(self, pergunta):
        mensagens = self._montar_mensagens(pergunta)
        resposta = self.llm.invoke(mensagens)
        texto = resposta.content
        self.historico.append({'role': 'user', 'content': pergunta})
        self.historico.append({'role': 'assistant', 'content': texto})
        return texto

    def resetar(self):
        self.historico.clear()
        print('Historico resetado.')


chatbot = ChargeGridChatbot(modelo='gpt-4o-mini', temperatura=0.3)
print(f'ChargeGrid Intelligence inicializado — modelo: {chatbot.modelo}')

## Célula 7 — Demonstração Interativa

Interface de conversa interativa com o assistente. Comandos disponíveis: `sair` encerra a sessão, `reset` limpa o histórico.


In [ ]:
chatbot.resetar()

print('=' * 62)
print(' ChargeGrid Intelligence — Assistente Gerencial')
print(' EV Challenge 2026 | GoodWe x FIAP')
print('=' * 62)

while True:
    try:
        entrada = input('\nOperador: ').strip()
    except (EOFError, KeyboardInterrupt):
        break

    if not entrada:
        continue
    if entrada.lower() == 'sair':
        print('Sistema encerrado.')
        break
    if entrada.lower() == 'reset':
        chatbot.resetar()
        continue

    resposta = chatbot.conversar(entrada)
    print(f'\nChargeGrid: {resposta}')

## Célula 8 — Validação: Matriz de Testes Sprint 1

Execução dos 5 casos de teste definidos na Sprint 1, mais 2 novos casos:
- **Teste 6:** Pergunta fora do escopo (valida recusa)
- **Teste 7:** Edge case — falha de hardware durante sessão ativa

In [ ]:
CASOS_DE_TESTE = [
    {
        'id': 1,
        'categoria': 'Controle de Demanda',
        'pergunta': (
            'Tenho 200 kW contratados na minha loja. Nas sextas a noite meu consumo '
            'chega a 170 kW com o movimento todo. Quantos carregadores consigo ligar '
            'ao mesmo tempo sem risco de multa?'
        ),
    },
    {
        'id': 2,
        'categoria': 'Faturamento e Precificacao',
        'pergunta': (
            'Ontem tive uma sessao de recarga que durou 45 minutos num carregador DC Fast. '
            'Como o sistema calcula o valor final que o cliente vai pagar e onde fica '
            'registrado isso para eu poder comprovar depois?'
        ),
    },
    {
        'id': 3,
        'categoria': 'Base Regulatoria',
        'pergunta': (
            'Tem algum problema com a lei se eu ficar mudando o preco da recarga toda hora '
            'dependendo do movimento do meu estacionamento?'
        ),
    },
    {
        'id': 4,
        'categoria': 'Dados Operacionais',
        'pergunta': (
            'Como eu tenho certeza de que o cliente ta pagando exatamente pelo que consumiu? '
            'De onde o sistema tira essa informacao na hora de fechar o caixa?'
        ),
    },
    {
        'id': 5,
        'categoria': 'Interoperabilidade',
        'pergunta': (
            'Se o negocio der certo e eu quiser comprar carregadores de outras marcas no ano '
            'que vem, esse sistema ainda vai conseguir gerenciar tudo junto?'
        ),
    },
    {
        'id': 6,
        'categoria': 'Fora do Escopo (out-of-scope)',
        'pergunta': (
            'Voce pode me recomendar um bom restaurante perto do meu shopping?'
        ),
    },
    {
        'id': 7,
        'categoria': 'Edge Case — Falha de Hardware',
        'pergunta': (
            'Um carregador parou de funcionar no meio de uma sessao e o cliente nao foi '
            'cobrado pelo total que consumiu. Como o sistema trata isso no faturamento?'
        ),
    },
]

bot_teste = ChargeGridChatbot(modelo='gpt-4o-mini', temperatura=0.2)
resultados = []

print('=' * 70)
print('VALIDACAO — MATRIZ DE TESTES SPRINT 1 (+ 2 NOVOS CASOS)')
print('ChargeGrid Intelligence | EV Challenge 2026')
print('=' * 70)

for tc in CASOS_DE_TESTE:
    print(f'\n{"─" * 70}')
    print(f'TESTE {tc["id"]} — {tc["categoria"]}')
    print(f'{"─" * 70}')
    print(f'Pergunta:\n{tc["pergunta"]}')
    resposta = bot_teste.conversar(tc['pergunta'])
    print(f'\nResposta ChargeGrid Intelligence:\n{resposta}')
    resultados.append({
        'id': tc['id'],
        'categoria': tc['categoria'],
        'pergunta': tc['pergunta'],
        'resposta': resposta,
    })

print(f'\n{"=" * 70}')
print(f'TESTES CONCLUIDOS: {len(resultados)}/7')
print('Validacao concluida: 7/7 testes executados.')
print('=' * 70)